# Day 1, step 1: is there a usable timestamp?

Before any time-based split, confirm Dreaddit has a real date for each post. Run `python src/download_data.py` first.

In [1]:
import pandas as pd

train = pd.read_csv("../data/raw/dreaddit_train.csv")
test = pd.read_csv("../data/raw/dreaddit_test.csv")
print(train.shape, test.shape)

# 116 columns, but most are pre-computed features (lex_liwc_*, lex_dal_*, syntax_*).
# We only care about the ones that describe the post itself.
core = ["subreddit", "post_id", "sentence_range", "text", "label", "confidence", "social_timestamp"]
train[core].head()

(2838, 116) (715, 116)


,subreddit,post_id,sentence_range,text,label,confidence,social_timestamp
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",1,0.8,1521614353
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",0,1.0,1527009817
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,1,0.8,1535935605
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",1,0.6,1516429555
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1,0.8,1539809005


In [2]:
# social_timestamp is Unix time: seconds since 1970-01-01 UTC.
for name, df in [("train", train), ("test", test)]:
    ts = pd.to_datetime(df["social_timestamp"], unit="s", utc=True)
    print(f"{name}: missing={ts.isna().sum()}  earliest={ts.min()}  latest={ts.max()}")

train: missing=0  earliest=2017-01-01 12:36:58+00:00  latest=2018-11-19 01:54:20+00:00
test: missing=0  earliest=2017-01-02 04:02:57+00:00  latest=2018-11-18 23:20:24+00:00


In [3]:
# A temporal split needs posts spread across time, not bunched into one week.
train["month"] = pd.to_datetime(train["social_timestamp"], unit="s", utc=True).dt.strftime("%Y-%m")
train.groupby("month").agg(rows=("label", "size"), stress_rate=("label", "mean"))

C:\Users\khana\AppData\Local\Temp\ipykernel_60808\1131746077.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["month"] = pd.to_datetime(train["social_timestamp"], unit="s", utc=True).dt.strftime("%Y-%m")


,rows,stress_rate
month,,
2017-01,88,0.670455
2017-02,46,0.565217
2017-03,90,0.488889
2017-04,51,0.431373
2017-05,60,0.600000
2017-06,63,0.587302
2017-07,76,0.592105
2017-08,67,0.641791
2017-09,69,0.550725


In [4]:
# One Reddit post can appear as several rows (different sentence_range chunks).
# Check that all chunks of a post share one timestamp, so a time split can't cut a post in half.
per_post = train.groupby("post_id")["social_timestamp"].nunique()
print("unique posts:", len(per_post), " posts with >1 row:", (train["post_id"].value_counts() > 1).sum())
print("posts whose rows have different timestamps:", (per_post > 1).sum())

unique posts: 2343  posts with >1 row: 380
posts whose rows have different timestamps: 0


# Day 1, step 2: what is the January 2018 spike?

January 2018 has ~650 train rows; most months have 50–180. Before choosing a cutoff date for Split B, find out what is in that month.

In [5]:
# Which subreddits make up the spike?
jan18 = train[train["month"] == "2018-01"]
print("Jan 2018 rows:", len(jan18), " unique posts:", jan18["post_id"].nunique())
jan18["subreddit"].value_counts()

Jan 2018 rows: 646  unique posts: 497


subreddit
relationships       552
anxiety              27
ptsd                 19
homeless             13
assistance           12
domesticviolence      8
survivorsofabuse      7
food_pantry           4
stress                2
almosthomeless        2
Name: count, dtype: int64

In [6]:
# Ruling out "one busy day": if a single event caused the spike, most rows would sit on 1-2 days.
# .dt.date drops the time of day, so value_counts() gives rows per calendar day (busiest first).
jan18_days = pd.to_datetime(jan18["social_timestamp"], unit="s", utc=True).dt.date
print("days in Jan 2018 with at least one row:", jan18_days.nunique())
jan18_days.value_counts().head(10)

days in Jan 2018 with at least one row: 31


social_timestamp
2018-01-23    36
2018-01-04    35
2018-01-28    34
2018-01-05    33
2018-01-29    29
2018-01-07    26
2018-01-13    25
2018-01-18    24
2018-01-30    22
2018-01-24    22
Name: count, dtype: int64

In [7]:
# Is that subreddit present in other months too? Count rows per subreddit per year, train and test.
# crosstab = a count table: one row per subreddit, one column per year.
for name, df in [("train", train), ("test", test)]:
    year = pd.to_datetime(df["social_timestamp"], unit="s", utc=True).dt.year
    print(f"--- {name} ---")
    print(pd.crosstab(df["subreddit"], year), "\n")

# Where in time does r/relationships appear at all?
rel_dates = pd.to_datetime(train.loc[train["subreddit"] == "relationships", "social_timestamp"], unit="s", utc=True)
print("r/relationships (train):", rel_dates.min(), "to", rel_dates.max())

--- train ---
social_timestamp  2017  2018
subreddit                   
almosthomeless      42    38
anxiety            199   304
assistance         119   170
domesticviolence   130   186
food_pantry          9    28
homeless            42   126
ptsd               218   366
relationships        0   552
stress              18    46
survivorsofabuse    95   150 

--- test ---
social_timestamp  2017  2018
subreddit                   
almosthomeless      12     7
anxiety             76    71
assistance          21    45
domesticviolence    28    44
food_pantry          3     3
homeless            18    34
ptsd                39    88
relationships        0   142
stress               4    10
survivorsofabuse    27    43 

r/relationships (train): 2018-01-01 05:36:27+00:00 to 2018-01-31 04:40:28+00:00
